In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1) # Regression راح تفك ضغطه عشان تطلع  مخرج
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

print(f"X_train_tensor : {X_train_tensor.shape}")
print(f"y_train_tensor : {y_train_tensor.shape}")
print(f"X_test_tensor : {X_test_tensor.shape}")
print(f"y_test_tensor : {y_test_tensor.shape}")



In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(f"Train dataset length: {len(train_dataset)}")
print(f"Test dataset length: {len(test_dataset)}")



In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train loader created with batch size: {batch_size}")
print(f"Test loader created with batch size: {batch_size}")



In [ ]:
# 4. Print shape of one batch
for X_batch, y_batch in train_loader:
    print(f"Shape of one batch from train_loader (X_batch): {X_batch.shape}")
    print(f"Shape of one batch from train_loader (y_batch): {y_batch.shape}")
    break


In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# Get one batch of training data
for X_batch, y_batch in train_loader:
    break

images_to_display = X_batch[:5].permute(0, 2, 3, 1).numpy()
ages_to_display = y_batch[:5].squeeze().numpy()

plt.figure(figsize=(12, 6))
for i in range(images_to_display.shape[0]):
    plt.subplot(1, 5, i + 1)
    plt.imshow(images_to_display[i])
    plt.title(f"Age: {int(ages_to_display[i])}")
    plt.axis('off')
plt.suptitle("Sample Images")
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class AgePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        input_size = 3 * 36 * 36
        hidden_size1 = 512
        hidden_size2 = 256
        hidden_size3 = 128
        output_size = 1  # لحساب توقع العمر

        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.fc4 = nn.Linear(hidden_size3, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Flatten the input image (N, C, H, W) to (N, C*H*W)
        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)  # No activation for the output layer in regression
        return x

print("AgePredictor model class defined successfully.")

In [ ]:
# Task 2: Write your training loop here:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Return average loss for the epoch
    return running_loss / len(train_loader)

print("train_epoch function defined.")

In [ ]:
# Task 3: Write your validation loop here:
def validate_epoch(model, test_loader, criterion, device):
    model.eval()  # Set the model here
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

    # Return average loss for the epoch
    return running_loss / len(test_loader)

print("validate_epoch function defined.")

In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.optim as optim

# 1. Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Create model instance and move to device
model = AgePredictor().to(device)
print("Model created and moved to device.")

# 3. Define loss function
criterion = nn.MSELoss()
print("Loss function (MSE) defined.")

# 4. Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Optimizer (Adam) defined.")

In [ ]:
# Task 5: Start training for 20 epochs:
epochs = 20
train_losses = []
val_losses = []

print("Starting training...")
for epoch in range(epochs):
    # Train for one epoch
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    # Validate for one epoch
    val_loss = validate_epoch(model, test_loader, criterion, device)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("Training complete.")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss over Epochs')
plt.legend()
plt.grid(True)
plt.show()

print("Loss plot generated successfully.")

In [ ]:
# Task 2 (Bonus): Write your code here: